# Apache Spark DataFrame 与 SQL 优化

> **高频考点** | Senior Data Engineer 面试准备

## 目录
1. Lazy Evaluation & Action vs Transformation
2. Cache / Persist 存储级别选择
3. Broadcast Join vs Sort-Merge Join
4. Bucketing 减少 Shuffle
5. Skew Join 倾斜处理（Salting）
6. Dynamic Partition Pruning
7. AQE 自适应查询执行
8. 练习题

---

## 1. Lazy Evaluation & Action vs Transformation

### 核心概念

**惰性求值 (Lazy Evaluation)**：Transformation 不立即执行，只记录计算步骤，直到遇到 Action 才触发实际计算。

```
Transformation (惰性)              Action (立即触发计算)
┌─────────────────────┐            ┌─────────────────────┐
│ map()               │            │ collect()           │
│ filter()            │            │ count()             │
│ flatMap()           │            │ show()              │
│ select()            │   ──────►  │ take(n)             │
│ groupBy()           │  触发计算   │ first()             │
│ join()              │            │ write.save()        │
│ withColumn()        │            │ foreach()           │
│ orderBy()           │            │ reduce()            │
└─────────────────────┘            └─────────────────────┘
     只构建 DAG                         触发 DAG 执行
```

### 惰性求值的优势
1. **优化机会**：Catalyst 可以看到完整的计算图，做全局优化
2. **避免不必要的计算**：如果 Action 只需要部分结果，可以提前终止
3. **流水线执行**：窄依赖可以不写磁盘直接流水线处理

In [ ]:
# 用 Python 生成器演示惰性求值的概念
# 生成器在 Python 中实现了类似 Spark Transformation 的惰性行为

import time

def lazy_source(n: int):
    """模拟数据源（类似 sc.parallelize 或 spark.read）"""
    print(f"  [Source] 准备产生 {n} 条数据...")
    for i in range(n):
        yield i

def lazy_map(source, fn, name="map"):
    """惰性 map（类似 Spark 的 Transformation）- 不立即执行"""
    # 注意：这里没有任何打印，因为函数体还没执行
    return (fn(x) for x in source)  # 返回生成器

def lazy_filter(source, predicate, name="filter"):
    """惰性 filter - 不立即执行"""
    return (x for x in source if predicate(x))

def action_collect(pipeline):
    """Action: collect - 触发实际计算"""
    print("  [Action: collect] 触发计算管道!")
    return list(pipeline)  # 这里才真正迭代

def action_count(pipeline):
    """Action: count - 触发实际计算"""
    print("  [Action: count] 触发计算管道!")
    return sum(1 for _ in pipeline)

print("=" * 55)
print("惰性求值演示")
print("=" * 55)

print("\n--- 构建计算管道（不执行任何计算）---")
source = lazy_source(10)
print("  调用 lazy_source(10)... [无输出 = 未执行]")
step1 = lazy_map(source, lambda x: x * 2)
print("  调用 lazy_map(x * 2)... [无输出 = 未执行]")
step2 = lazy_filter(step1, lambda x: x > 8)
print("  调用 lazy_filter(x > 8)... [无输出 = 未执行]")
print("  管道已建立，但没有数据流动!")

print("\n--- 执行 Action，触发计算 ---")
result = action_collect(step2)
print(f"  结果: {result}")

print("\n" + "-" * 55)
print("\n优化示例: 只需要第一个元素 → 不需要处理所有数据")

def action_first(pipeline):
    """Action: first - 只需要第一个元素"""
    print("  [Action: first] 触发计算管道!")
    return next(iter(pipeline), None)  # 只取第一个就停止

def lazy_source_verbose(n: int):
    for i in range(n):
        print(f"  [Source] 生成第 {i} 条数据")
        yield i

source2 = lazy_source_verbose(10)
step2a = lazy_filter(source2, lambda x: x > 5)
first_result = action_first(step2a)
print(f"  结果: {first_result} (仅处理到找到第一个满足条件的元素)")

In [ ]:
# PySpark Transformation vs Action 对比代码
# (需要 PySpark 环境运行，这里展示代码和解释)

PYSPARK_LAZY_EVAL = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper
import time

spark = SparkSession.builder.appName("LazyEval").getOrCreate()

# 所有这些都是 Transformation，立即返回，不执行计算
start = time.time()

df = spark.range(0, 10_000_000)                    # Transformation
df2 = df.withColumn("doubled", col("id") * 2)      # Transformation
df3 = df2.filter(col("doubled") > 1_000_000)       # Transformation
df4 = df3.select("doubled")                        # Transformation

print(f"构建 DAG 耗时: {time.time()-start:.4f}s")  # 几乎为 0

# Action 才触发实际计算
start = time.time()
count = df4.count()                                 # ACTION - 触发计算
print(f"count() 耗时: {time.time()-start:.2f}s")
print(f"结果: {count}")

# 每次 Action 都会重新计算（如果没有 cache）
start = time.time()
first = df4.first()                                 # ACTION - 再次触发计算
print(f"first() 耗时: {time.time()-start:.2f}s")
'''

print("PySpark 惰性求值代码示例:")
print(PYSPARK_LAZY_EVAL)

print("="*55)
print("关键点总结:")
print("="*55)
key_points = [
    "Transformation 返回新 DataFrame/RDD，不执行计算",
    "Action 触发 DAG 执行，返回结果给 Driver",
    "每次 Action 默认都会重新计算（除非 cache/persist）",
    "Catalyst 可以看到完整 DAG 后做全局优化",
    "惰性求值允许: 谓词下推、列裁剪、Join 重排序等优化",
]
for i, point in enumerate(key_points, 1):
    print(f"  {i}. {point}")

## 2. Cache / Persist 存储级别选择

### 存储级别对比

| 存储级别 | 内存 | 磁盘 | 副本数 | 适用场景 |
|---------|------|------|--------|----------|
| `MEMORY_ONLY` | 反序列化 | 否 | 1 | 内存充足，需要最快访问 |
| `MEMORY_ONLY_SER` | 序列化 | 否 | 1 | 内存紧张，CPU 充足 |
| `MEMORY_AND_DISK` | 反序列化 | 溢出 | 1 | **最常用**，内存不足时溢出到磁盘 |
| `MEMORY_AND_DISK_SER` | 序列化 | 溢出 | 1 | 内存紧张但需要持久化 |
| `DISK_ONLY` | 否 | 是 | 1 | 内存极度紧张 |
| `MEMORY_ONLY_2` | 反序列化 | 否 | 2 | 高可用，内存充足 |
| `OFF_HEAP` | 堆外内存 | 否 | 1 | 减少 GC 压力，Tungsten |

### 什么时候使用 Cache？

```
决策树:

数据是否被多个 Action 使用?
├── 否 → 不需要 Cache (重算比 I/O 便宜)
└── 是 → 数据量 vs 内存大小?
          ├── 数据 < 内存  → MEMORY_ONLY
          ├── 数据 ≈ 内存  → MEMORY_AND_DISK (推荐)
          └── 数据 >> 内存 → DISK_ONLY 或不 Cache
                            (反复读磁盘可能不如重算)
```

### Cache vs Persist
- `df.cache()` = `df.persist(StorageLevel.MEMORY_AND_DISK)` (DataFrame默认)
- `rdd.cache()` = `rdd.persist(StorageLevel.MEMORY_ONLY)` (RDD默认)
- `persist()` 允许指定存储级别

### 重要注意事项
- Cache 是**惰性**的：需要 Action 触发才真正缓存
- 用完记得 `unpersist()`，否则占用内存
- LRU 驱逐：内存不足时自动驱逐最久未使用的分区

In [ ]:
# 用 Python 模拟 Cache 的效果
# 展示有无 Cache 时的计算次数差异

computation_count = 0

def expensive_computation(data):
    """模拟耗时的 ETL 计算"""
    global computation_count
    computation_count += 1
    print(f"  [计算] 第 {computation_count} 次执行耗时计算...")
    # 模拟复杂转换
    return [(x * 2, x ** 2) for x in data if x % 2 == 0]

raw_data = list(range(10))

print("=" * 55)
print("场景 1: 没有 Cache - 每次 Action 都重新计算")
print("=" * 55)
computation_count = 0

# 等价于 Spark 的惰性 DataFrame（未 cache）
def get_df():  # 每次调用都重新计算
    return expensive_computation(raw_data)

# Action 1: count
count = len(get_df())
print(f"  count = {count}")

# Action 2: first
first = get_df()[0]
print(f"  first = {first}")

# Action 3: collect for analysis
all_data = get_df()
total = sum(x[0] for x in all_data)
print(f"  sum = {total}")

print(f"\n总计算次数: {computation_count} 次 (每个 Action 都重新计算!)") 

print()
print("=" * 55)
print("场景 2: 使用 Cache - 计算一次，复用结果")
print("=" * 55)
computation_count = 0

# 模拟 df.cache() + 第一次 Action 触发缓存
cached_df = None  # None = 未缓存

def get_cached_df():
    global cached_df
    if cached_df is None:
        cached_df = expensive_computation(raw_data)  # 只计算一次
    else:
        print("  [Cache] 从缓存读取，跳过计算")
    return cached_df

# Action 1: count (触发缓存)
count = len(get_cached_df())
print(f"  count = {count}")

# Action 2: first (从缓存读)
first = get_cached_df()[0]
print(f"  first = {first}")

# Action 3: collect (从缓存读)
all_data = get_cached_df()
total = sum(x[0] for x in all_data)
print(f"  sum = {total}")

print(f"\n总计算次数: {computation_count} 次 (只计算一次!)")
print("\n结论: 当同一 DataFrame 被多个 Action 使用时，Cache 效果显著")

## 3. Broadcast Join vs Sort-Merge Join

### Broadcast Hash Join

```
条件: 小表大小 < spark.sql.autoBroadcastJoinThreshold (默认 10MB)

                    Driver 收集小表
                         │
                    ┌────▼────┐
                    │ 小表数据 │ (例: countries 表, 5MB)
                    └────┬────┘
                         │ 广播到所有 Executor
           ┌─────────────┼─────────────┐
           ▼             ▼             ▼
    ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
    │ Executor 1  │ │ Executor 2  │ │ Executor 3  │
    │ [小表副本]   │ │ [小表副本]   │ │ [小表副本]   │
    │ + 大表分区1  │ │ + 大表分区2  │ │ + 大表分区3  │
    │ → Join      │ │ → Join      │ │ → Join      │
    └─────────────┘ └─────────────┘ └─────────────┘
    大表无需 Shuffle!  大表无需 Shuffle!  大表无需 Shuffle!

优点: 大表完全不需要 Shuffle
缺点: 小表复制到每个 Executor (内存开销)
```

### Sort-Merge Join

```
条件: 两表都很大，无法广播

阶段 1 - Shuffle (按 Join Key 分区):
┌──────────┐                    ┌──────────┐
│ 大表 A    │──[Shuffle by key]──►│ 分区 0   │
│ 大表 B    │──[Shuffle by key]──►│ key=1,2  │
└──────────┘                    └──────────┘
                                ┌──────────┐
                                │ 分区 1   │
                                │ key=3,4  │
                                └──────────┘

阶段 2 - Sort (每个分区内部排序):
   A的分区0: [1, 1, 2] (排序后)
   B的分区0: [1, 2, 2] (排序后)

阶段 3 - Merge (双指针合并):
   A[0]=1, B[0]=1 → Match! 输出
   A[0]=1, B[1]=2 → A 前进
   A[1]=2, B[1]=2 → Match! 输出

优点: 可以处理任意大小的表
缺点: 需要 Shuffle + Sort，代价高
```

### Join 策略选择

| 策略 | 条件 | 优势 |
|------|------|------|
| BroadcastHashJoin | 小表 < 10MB | 无大表 Shuffle |
| ShuffleHashJoin | 一表较小，内存能放下哈希表 | 无排序开销 |
| SortMergeJoin | 两表都很大 | 可扩展，适合超大表 |
| CartesianProduct | 无 Join 条件 | N/A（避免！）|
| BroadcastNestedLoop | 不等值 Join | 最慢（避免！）|

In [ ]:
# 用 Pandas 演示 Broadcast Join vs Sort-Merge Join 的逻辑
import pandas as pd
import numpy as np
import time

np.random.seed(42)

# 模拟大表（orders）和小表（country_info）
large_table = pd.DataFrame({
    'order_id': range(100_000),
    'country_code': np.random.choice(['US', 'CN', 'UK', 'DE', 'JP'], 100_000),
    'amount': np.random.randint(10, 1000, 100_000)
})

small_table = pd.DataFrame({
    'country_code': ['US', 'CN', 'UK', 'DE', 'JP'],
    'country_name': ['United States', 'China', 'United Kingdom', 'Germany', 'Japan'],
    'tax_rate': [0.08, 0.13, 0.20, 0.19, 0.10]
})

print("=" * 55)
print("Join 策略对比")
print("=" * 55)
print(f"\n大表 (orders): {len(large_table):,} 行, ~{large_table.memory_usage(deep=True).sum()//1024}KB")
print(f"小表 (country_info): {len(small_table)} 行, ~{small_table.memory_usage(deep=True).sum()}B")

# 模拟 Broadcast Hash Join
# 小表构建哈希表，大表每个分区本地查找
print("\n[Broadcast Hash Join 模拟]")
print("步骤 1: 将小表广播到所有 Executor (构建哈希表)")
broadcast_dict = small_table.set_index('country_code').to_dict('index')  # 哈希表
print(f"  哈希表大小: {len(broadcast_dict)} 条记录")
print("步骤 2: 大表每行本地查找哈希表 (无需 Shuffle)")

start = time.time()
# 向量化哈希查找
result_broadcast = large_table.merge(small_table, on='country_code', how='inner')
time_broadcast = time.time() - start
print(f"  结果: {len(result_broadcast):,} 行, 耗时: {time_broadcast*1000:.2f}ms")
print(f"  (大表分区无需移动! 小表广播到每个 Executor)")

# 模拟 Sort-Merge Join
# 两表按 key shuffle 分区，然后排序，双指针合并
print("\n[Sort-Merge Join 模拟]")
print("步骤 1: 两表按 Join Key Shuffle (模拟分区)")

# 模拟 shuffle: 按 country_code 排序
start = time.time()
sorted_large = large_table.sort_values('country_code')  # Shuffle + Sort
sorted_small = small_table.sort_values('country_code')  # Shuffle + Sort
print("步骤 2: 各分区内部排序")
print("步骤 3: 双指针合并 (Merge Join)")
result_smj = sorted_large.merge(sorted_small, on='country_code', how='inner')
time_smj = time.time() - start
print(f"  结果: {len(result_smj):,} 行, 耗时: {time_smj*1000:.2f}ms")
print(f"  (需要 Shuffle + Sort，代价较高)")

print(f"\n结论: 小表 Broadcast 比 Sort-Merge 快 {time_smj/time_broadcast:.1f}x")
print("实际 Spark 中差距更大（因为 Sort-Merge 需要网络 Shuffle）")

In [ ]:
# PySpark 中强制 Broadcast Join 的代码示例

PYSPARK_BROADCAST = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col

spark = SparkSession.builder.appName("JoinDemo").getOrCreate()

# 设置广播阈值 (默认 10MB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50mb")

orders = spark.table("orders")           # 大表 (100GB)
country_info = spark.table("countries")  # 小表 (1MB)

# 方法 1: 自动广播 (如果表小于阈值，Catalyst 自动选择)
result1 = orders.join(country_info, "country_code")

# 方法 2: 强制广播 Hint
result2 = orders.join(broadcast(country_info), "country_code")

# 方法 3: SQL Hint
# spark.sql("""
#     SELECT /*+ BROADCAST(c) */ o.*, c.country_name
#     FROM orders o JOIN countries c ON o.country_code = c.code
# """)

# 禁用广播 (强制 Sort-Merge Join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 查看执行计划
result2.explain()  # 应该看到 BroadcastHashJoin
'''

print("PySpark Broadcast Join 代码示例:")
print(PYSPARK_BROADCAST)

print("\n注意事项:")
print("  1. autoBroadcastJoinThreshold 默认 10MB（压缩前的估计大小）")
print("  2. 广播太大的表会 OOM（小心！）")
print("  3. broadcast() Hint 是强制的，即使表很大也会广播")
print("  4. AQE 可以在运行时动态切换到 Broadcast Join")

## 4. Bucketing 减少 Shuffle

### Bucketing 原理

Bucketing 是一种**预分区**技术：将数据按指定列的 Hash 值分成固定数量的 Bucket，存储到 HDFS 上。

```
写入时 (一次性开销):
┌──────────┐   hash(user_id) % 16   ┌──────────────────────────┐
│  数据     │ ──────────────────────► │ bucket_00001.parquet      │
│          │                         │ bucket_00002.parquet      │
└──────────┘                         │ ...                       │
                                     │ bucket_00016.parquet      │
                                     └──────────────────────────┘

查询时 (每次受益):
┌────────────────────────────────────────────────────────┐
│ SELECT * FROM orders o JOIN users u ON o.user_id=u.id  │
│                                                        │
│ 两表相同 bucket 的数据直接 join，不需要 Shuffle!        │
│                                                        │
│  orders_bucket_01  ←→  users_bucket_01  → Join         │
│  orders_bucket_02  ←→  users_bucket_02  → Join         │
│  ...               ←→  ...              → Join         │
└────────────────────────────────────────────────────────┘
```

### Bucketing vs Partitioning 对比

| 特性 | Partitioning | Bucketing |
|------|-------------|----------|
| 目的 | 数据剪枝（过滤） | 减少 Join/GroupBy Shuffle |
| 实现 | 按列值分目录 | 按 Hash 分文件 |
| 适用列 | 低基数列（日期、地区） | 高基数列（user_id、order_id） |
| 查询优化 | WHERE 条件过滤 | Join/GroupBy 消除 Shuffle |
| 文件结构 | `/date=2024-01/` 目录 | `part-r-00001-*.parquet` 文件 |

### 使用 Bucketing 的条件
1. 两个大表频繁按同一列 Join
2. Bucket 数量相同（或一个是另一个的整数倍）
3. 表相对稳定（写入频率低，查询频率高）

In [ ]:
# 模拟 Bucketing 的效果
# 展示预分区数据如何消除 Join 时的 Shuffle

import hashlib
from collections import defaultdict

NUM_BUCKETS = 4  # 模拟 4 个 bucket

def bucket_id(key: int, num_buckets: int) -> int:
    """模拟 Spark 的 bucket hash 函数"""
    return hash(key) % num_buckets

# 模拟已 Bucket 存储的数据
orders_raw = [
    {"order_id": i, "user_id": i % 10, "amount": i * 100}
    for i in range(20)
]

users_raw = [
    {"user_id": i, "name": f"user_{i}"}
    for i in range(10)
]

# 写入时按 user_id 分 bucket
orders_bucketed = defaultdict(list)
users_bucketed = defaultdict(list)

for order in orders_raw:
    bid = bucket_id(order["user_id"], NUM_BUCKETS)
    orders_bucketed[bid].append(order)

for user in users_raw:
    bid = bucket_id(user["user_id"], NUM_BUCKETS)
    users_bucketed[bid].append(user)

print("=" * 55)
print("Bucketing 效果演示")
print("=" * 55)
print(f"\n数据写入时按 user_id Bucket (NUM_BUCKETS={NUM_BUCKETS}):")
for bid in range(NUM_BUCKETS):
    o_keys = [o["user_id"] for o in orders_bucketed[bid]]
    u_keys = [u["user_id"] for u in users_bucketed[bid]]
    print(f"  Bucket {bid}: orders.user_id={o_keys}, users.user_id={u_keys}")

print("\nJoin 时（无需 Shuffle！）:")
print("相同 bucket_id 的数据直接 join，因为 hash 值相同")

total_results = []
for bid in range(NUM_BUCKETS):
    # 每个 bucket 内部 join（本地操作，无网络传输）
    users_map = {u["user_id"]: u["name"] for u in users_bucketed[bid]}
    bucket_results = []
    for order in orders_bucketed[bid]:
        user_id = order["user_id"]
        if user_id in users_map:
            bucket_results.append({
                "order_id": order["order_id"],
                "user_name": users_map[user_id],
                "amount": order["amount"]
            })
    total_results.extend(bucket_results)
    print(f"  Bucket {bid}: 本地 join 产生 {len(bucket_results)} 条结果 (无 Shuffle!)")

print(f"\n最终结果: {len(total_results)} 条")

print()
PYSPARK_BUCKETING = '''
# PySpark Bucketing 代码
# 写入时指定 bucket
orders.write \\
    .bucketBy(16, "user_id") \\
    .sortBy("user_id") \\
    .saveAsTable("orders_bucketed")

users.write \\
    .bucketBy(16, "user_id") \\
    .sortBy("user_id") \\
    .saveAsTable("users_bucketed")

# 查询时自动消除 Shuffle
result = spark.table("orders_bucketed").join(
    spark.table("users_bucketed"), "user_id"
)
result.explain()  # 不会看到 Exchange (Shuffle) 算子
'''
print("PySpark Bucketing 代码:")
print(PYSPARK_BUCKETING)

## 5. Skew Join 倾斜处理（Salting）

### 数据倾斜的表现

```
正常情况 (数据均匀):
  Task 0: ████████ 1000 行
  Task 1: ████████ 1100 行
  Task 2: ████████  900 行
  Task 3: ████████ 1000 行
  完成时间: ~10s (均衡)

数据倾斜 (某个 key 特别多):
  Task 0: ████████ 1000 行                    ← 完成 10s
  Task 1: ████████████████████████ 50000 行   ← 完成 500s (!)  
  Task 2: ████████ 900 行                     ← 完成 9s
  Task 3: ████████ 1000 行                    ← 完成 10s
  整个 Stage 等待 Task 1: 500s (长尾任务)
```

### Salting 解决方案

```
原始数据 (hot key = "apple" 占 90% 数据):

  热点一侧 (大表):                 非热点一侧 (小表):
  apple_1, apple_2, ..., apple_N   apple, ...其他

Salting 步骤:

  Step 1: 大表的热点 key 加随机前缀 (0-N)
    apple → apple_0, apple_1, apple_2, ..., apple_9
    (根据 SALT_FACTOR=10 分散到 10 个分区)

  Step 2: 小表的对应 key 复制 N 份
    apple → apple_0, apple_1, ..., apple_9 (10 份)

  Step 3: 按 salted_key Join
    apple_0 ←→ apple_0 (1/10 数据量)
    apple_1 ←→ apple_1 (1/10 数据量)
    ...每个 Task 只处理 1/10 的热点数据
```

In [ ]:
# 用 Python 实现 Salting 技术演示
import pandas as pd
import numpy as np
import random
import time

np.random.seed(42)
random.seed(42)

SALT_FACTOR = 10  # 将热点 key 分散到 10 个分区

# 生成倾斜数据
# 90% 的 orders 来自 category='Electronics'（热点 key）
n_orders = 10_000
categories = (['Electronics'] * int(n_orders * 0.9) + 
              random.choices(['Clothing', 'Food', 'Books', 'Sports'], k=int(n_orders * 0.1)))
random.shuffle(categories)

orders = pd.DataFrame({
    'order_id': range(n_orders),
    'category': categories,
    'amount': np.random.randint(10, 1000, n_orders)
})

category_info = pd.DataFrame({
    'category': ['Electronics', 'Clothing', 'Food', 'Books', 'Sports'],
    'tax_rate': [0.08, 0.05, 0.03, 0.05, 0.08]
})

print("=" * 55)
print("Salting 处理数据倾斜")
print("=" * 55)
print("\n数据分布:")
print(orders['category'].value_counts().to_string())

# --- 方法 1: 普通 Join（有倾斜）---
print("\n[方法 1] 普通 Join（数据倾斜）:")
# 模拟分区分配（按 hash 分到 5 个分区）
num_partitions = 5
orders_with_partition = orders.copy()
orders_with_partition['partition'] = orders_with_partition['category'].apply(
    lambda x: hash(x) % num_partitions
)
partition_sizes = orders_with_partition.groupby('partition').size()
print("  各分区数据量 (严重倾斜):")
for pid, size in partition_sizes.items():
    bar = '█' * (size // 100)
    print(f"    分区 {pid}: {bar} {size} 行")

# --- 方法 2: Salting Join ---
print(f"\n[方法 2] Salting Join (SALT_FACTOR={SALT_FACTOR}):")

# Step 1: 大表（orders）加随机 salt
orders_salted = orders.copy()
orders_salted['salt'] = np.random.randint(0, SALT_FACTOR, len(orders))
orders_salted['salted_category'] = orders_salted['category'] + '_' + orders_salted['salt'].astype(str)

# Step 2: 小表（category_info）扩展 SALT_FACTOR 倍
category_info_exploded = pd.concat([
    category_info.assign(salt=i, 
                         salted_category=category_info['category'] + '_' + str(i))
    for i in range(SALT_FACTOR)
], ignore_index=True)

# Step 3: 按 salted_category Join
result_salted = orders_salted.merge(category_info_exploded, on='salted_category')

# 检查倾斜是否消除
result_salted['partition'] = result_salted['salted_category'].apply(
    lambda x: hash(x) % (num_partitions * SALT_FACTOR)
)
# 简化：按 salt 值看分布
print(f"  小表扩展: {len(category_info)} 行 → {len(category_info_exploded)} 行 (复制 {SALT_FACTOR} 份)")

electronics_salt_dist = (
    orders_salted[orders_salted['category'] == 'Electronics']['salt'].value_counts().sort_index()
)
print(f"  Electronics (热点key) 按 salt 分布 (应该均匀):")
for salt_val, count in electronics_salt_dist.items():
    bar = '█' * (count // 50)
    print(f"    salt={salt_val}: {bar} {count} 行")

print(f"\n结果验证:")
print(f"  Join 结果行数: {len(result_salted):,} (应与原始 orders 相同: {len(orders):,})")
print(f"  热点 key 已从 {int(n_orders*0.9):,} 行分散到 {SALT_FACTOR} 个分区")
print(f"  每个分区约 {int(n_orders*0.9)//SALT_FACTOR:,} 行 (解决倾斜!)")

## 6. Dynamic Partition Pruning

### 概念

**Dynamic Partition Pruning (DPP)** 是 Spark 3.0 引入的优化，允许在运行时根据 Join 的一侧动态过滤另一侧的分区。

```
查询:
SELECT * FROM fact_sales fs
JOIN dim_date dd ON fs.date_id = dd.date_id
WHERE dd.year = 2024

传统执行 (无 DPP):
  1. 扫描 fact_sales 所有分区 (3 年的数据 = 1000 个分区)
  2. 扫描 dim_date，过滤 year=2024
  3. Join

有 DPP 的执行:
  1. 扫描 dim_date，过滤 year=2024，得到 date_id 列表
             ↓ 动态生成过滤条件
  2. 扫描 fact_sales，只扫描对应的 date_id 分区 (365 个分区)
  3. Join

        dim_date (year=2024)                fact_sales
        ┌─────────────────┐                ┌─────────────────────┐
        │ date_id: [1..365]│ ──────────────►│ 只扫描 2024 年分区   │
        └─────────────────┘   动态过滤      │ (跳过 2022, 2023)    │
                                            └─────────────────────┘
                                              节省 66% 的扫描量!
```

### DPP 生效条件
1. `spark.sql.optimizer.dynamicPartitionPruning.enabled = true` (默认开启)
2. Join 的过滤侧是小表（可以广播）
3. 被过滤的表按 Join Key 分区
4. 等值 Join（不支持范围 Join）

In [ ]:
# 用 DuckDB 演示 Dynamic Partition Pruning 的效果
# DuckDB 有类似的过滤下推机制

try:
    import duckdb
    import pandas as pd
    import numpy as np
    
    con = duckdb.connect()
    
    # 创建维度表和事实表
    np.random.seed(42)
    
    # 维度表：dim_date
    dates = pd.DataFrame({
        'date_id': range(1, 366 * 3 + 1),  # 3 年数据
        'year': [2022] * 365 + [2023] * 365 + [2024] * 365,
        'month': list(range(1, 13)) * 91 + [1] * 3,
        'quarter': [1]*90 + [2]*91 + [3]*92 + [4]*92  # 简化
    })
    dates = dates.iloc[:365*3]  # 修正长度
    
    # 事实表：fact_sales (按 date_id 分区)
    n_sales = 100_000
    sales = pd.DataFrame({
        'sale_id': range(n_sales),
        'date_id': np.random.randint(1, 366*3+1, n_sales),
        'amount': np.random.uniform(10, 1000, n_sales).round(2)
    })
    
    con.register('dim_date', dates)
    con.register('fact_sales', sales)
    
    print("=" * 55)
    print("Dynamic Partition Pruning 效果演示 (DuckDB)")
    print("=" * 55)
    print(f"fact_sales 总行数: {len(sales):,}")
    print(f"dim_date 总行数: {len(dates):,} (3年 = 1095天)")
    
    # 查询 1: 有 DPP 效果 (过滤维度表后影响事实表扫描)
    result = con.execute("""
        SELECT dd.year, COUNT(*) as sales_count, SUM(fs.amount) as total
        FROM fact_sales fs
        JOIN dim_date dd ON fs.date_id = dd.date_id
        WHERE dd.year = 2024
        GROUP BY dd.year
    """).fetchdf()
    
    print(f"\n查询结果 (WHERE dd.year=2024):")
    print(result.to_string(index=False))
    
    # 展示 DPP 的原理
    year_2024_date_ids = set(dates[dates['year'] == 2024]['date_id'])
    matching_sales = sales[sales['date_id'].isin(year_2024_date_ids)]
    print(f"\n手动模拟 DPP:")
    print(f"  2024年的 date_id 范围: {min(year_2024_date_ids)} - {max(year_2024_date_ids)}")
    print(f"  2024年对应的销售记录: {len(matching_sales):,} 行")
    print(f"  跳过的记录: {len(sales) - len(matching_sales):,} 行")
    print(f"  节省扫描: {(1 - len(matching_sales)/len(sales))*100:.1f}%")
    print(f"\nDPP 在 Spark 中工作原理:")
    print(f"  1. 先扫描 dim_date，得到满足条件的 date_id")
    print(f"  2. 将 date_id 作为动态过滤器应用到 fact_sales 的分区扫描")
    print(f"  3. 只读取 2024年 的分区，跳过 2022/2023 的分区")
    
    con.close()
    
except ImportError:
    print("DuckDB 未安装，展示概念说明:")
    print("""
Dynamic Partition Pruning (DPP) 核心概念:

  场景: SELECT * FROM fact JOIN dim WHERE dim.year = 2024

  传统做法:
    Scan(fact, all_partitions=1000) → Filter after join

  DPP 做法:
    1. Scan(dim, filter=year=2024) → 得到 date_id 集合
    2. Scan(fact, filter=date_id IN set) → 只扫描相关分区
    3. Join → 结果相同，但 fact 扫描量从 1000 分区降到 365 分区

  配置: spark.sql.optimizer.dynamicPartitionPruning.enabled=true (默认)
    """)

## 7. AQE 自适应查询执行

### AQE (Adaptive Query Execution) 三大功能

**Spark 3.0 引入，Spark 3.2 默认开启**（`spark.sql.adaptive.enabled=true`）

```
传统静态计划:                    AQE 动态调整:

编译时确定执行计划               运行时根据实际数据调整
        ↓                                ↓
Stage 0 → Stage 1 → Stage 2      Stage 0 完成后 →
                                    Shuffle 统计 →
预估数据量: 1GB                       实际: 100MB!
计划: SortMergeJoin                AQE 重新规划 →
实际: 表很小，应该 BroadcastJoin     改为 BroadcastHashJoin
```

### 功能 1: 自动合并小分区 (Coalesce Partitions)
```
Shuffle 后 (原始 200 分区):              AQE 合并后:
分区 0:  5MB  ████                       分区 0: 51MB ████████████████████████
分区 1:  3MB  ██                         (合并 0-9)
分区 2:  8MB  ████                       分区 1: 47MB ███████████████████████
...                                      (合并 10-19)
分区 199: 2MB █                          ...
→ 200 个小分区，调度开销大               → 少量大分区，更高效
```

### 功能 2: 动态切换 Join 策略
```
静态估计: 表A = 500MB → SortMergeJoin
实际运行: 过滤后只有 5MB → AQE 切换为 BroadcastHashJoin
```

### 功能 3: 自动处理数据倾斜 (Skew Join Optimization)
```
检测: 某分区比中位数大 5 倍 → 判定为倾斜
处理: 将倾斜分区拆分，另一侧对应数据复制
```

In [ ]:
# AQE 配置参数总结和效果演示

aqe_configs = [
    {
        "config": "spark.sql.adaptive.enabled",
        "default": "true (Spark 3.2+)",
        "description": "开启 AQE 总开关",
        "recommended": "true"
    },
    {
        "config": "spark.sql.adaptive.coalescePartitions.enabled",
        "default": "true",
        "description": "自动合并小 Shuffle 分区",
        "recommended": "true"
    },
    {
        "config": "spark.sql.adaptive.advisoryPartitionSizeInBytes",
        "default": "64MB",
        "description": "合并后目标分区大小",
        "recommended": "64MB~128MB"
    },
    {
        "config": "spark.sql.adaptive.coalescePartitions.minPartitionNum",
        "default": "1",
        "description": "合并后最少保留的分区数",
        "recommended": "= Executor 总核数"
    },
    {
        "config": "spark.sql.adaptive.skewJoin.enabled",
        "default": "true",
        "description": "自动处理 Join 数据倾斜",
        "recommended": "true"
    },
    {
        "config": "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
        "default": "5",
        "description": "分区超过中位数 N 倍时判定为倾斜",
        "recommended": "3~5"
    },
    {
        "config": "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
        "default": "256MB",
        "description": "分区超过此大小且满足 Factor 才判定倾斜",
        "recommended": "128MB"
    },
    {
        "config": "spark.sql.autoBroadcastJoinThreshold",
        "default": "10MB",
        "description": "小于此大小自动广播（AQE 使用 Shuffle 后的实际大小）",
        "recommended": "10MB~100MB"
    },
]

print("=" * 75)
print("AQE 关键配置参数")
print("=" * 75)
print(f"{'配置项':<55} {'默认值':<15} {'推荐值'}")
print("-" * 75)
for cfg in aqe_configs:
    print(f"{cfg['config']:<55} {cfg['default']:<15} {cfg['recommended']}")
    print(f"  说明: {cfg['description']}")

print()
print("=" * 55)
print("AQE 分区合并效果模拟")
print("=" * 55)

import random
random.seed(42)

# 模拟 Shuffle 后的分区大小（MB）
num_shuffle_partitions = 200
partition_sizes_mb = [random.uniform(0.1, 10) for _ in range(190)] + \
                     [random.uniform(50, 150) for _ in range(10)]
random.shuffle(partition_sizes_mb)

target_size_mb = 64  # advisoryPartitionSizeInBytes = 64MB

print(f"\nShuffle 后: {num_shuffle_partitions} 个分区")
print(f"  小分区 (<10MB): {sum(1 for s in partition_sizes_mb if s < 10)} 个")
print(f"  大分区 (>50MB): {sum(1 for s in partition_sizes_mb if s > 50)} 个")
print(f"  总数据量: {sum(partition_sizes_mb):.1f} MB")

# 模拟 AQE 合并小分区
coalesced = []
current_group_size = 0
for size in sorted(partition_sizes_mb):
    if current_group_size + size <= target_size_mb:
        current_group_size += size
    else:
        if current_group_size > 0:
            coalesced.append(current_group_size)
        current_group_size = size
if current_group_size > 0:
    coalesced.append(current_group_size)

print(f"\nAQE 合并后: {len(coalesced)} 个分区 (减少了 {num_shuffle_partitions - len(coalesced)} 个)")
print(f"  平均分区大小: {sum(coalesced)/len(coalesced):.1f} MB")
print(f"  调度开销减少: {(num_shuffle_partitions - len(coalesced))/num_shuffle_partitions*100:.0f}%")

## 复习要点

| 概念 | 关键记忆点 |
|------|------------|
| **Lazy Evaluation** | Transformation 不执行，Action 触发；每次 Action 默认重新计算 |
| **Cache** | 多个 Action 共用时使用；`cache()` = MEMORY_AND_DISK (DataFrame) |
| **Broadcast Join** | 小表 < 10MB 广播，大表无需 Shuffle |
| **Sort-Merge Join** | 两大表 Shuffle + Sort + Merge；代价高但可扩展 |
| **Bucketing** | 预分区，Join 时无 Shuffle；适合高频大表 Join |
| **Salting** | 热点 key 加随机前缀，小表复制 N 份；解决数据倾斜 |
| **DPP** | 维度表过滤条件动态传播到事实表分区裁剪 |
| **AQE** | 运行时自动合并小分区、切换 Join 策略、处理倾斜 |

---

## 练习题

---

### 练习 1：Transformation vs Action 判断

判断以下操作是 Transformation 还是 Action，并说明理由：

```python
df.select("name", "age")      # ?
df.count()                    # ?
df.filter(col("age") > 18)    # ?
df.show()                     # ?
df.write.parquet("/path")     # ?
df.groupBy("city").agg(...)   # ?
df.cache()                    # ?
df.take(10)                   # ?
```

In [ ]:
# 练习 1 答案

answers = [
    ("df.select('name', 'age')", "Transformation", "返回新 DataFrame，不执行计算"),
    ("df.count()",               "Action",          "返回 Long 值，触发计算"),
    ("df.filter(col('age')>18)", "Transformation", "返回过滤后的 DataFrame"),
    ("df.show()",                "Action",          "触发计算并打印结果到 Driver"),
    ("df.write.parquet('/path')","Action",          "触发计算并写入存储系统"),
    ("df.groupBy('city').agg()", "Transformation", "返回聚合 DataFrame"),
    ("df.cache()",               "Transformation*", "标记为缓存，但需 Action 才真正缓存"),
    ("df.take(10)",              "Action",          "返回 List[Row] 到 Driver"),
]

print(f"{'操作':<35} {'类型':<16} 说明")
print("-" * 80)
for op, op_type, reason in answers:
    icon = "⚡" if op_type == "Action" else "→ "
    print(f"{op:<35} {icon}{op_type:<14} {reason}")

### 练习 2：Cache 使用场景分析

以下代码是否应该使用 Cache？为什么？

```python
# 场景 A
df = spark.read.parquet("/data/sales")  # 500GB
result = df.filter(col("year") == 2024).count()

# 场景 B
df = spark.read.parquet("/data/users")   # 2GB
count = df.count()
avg_age = df.agg({"age": "avg"}).collect()
top_cities = df.groupBy("city").count().take(10)

# 场景 C
df = spark.read.parquet("/data/logs")    # 100GB
for date in date_list:  # 30 次循环
    daily = df.filter(col("date") == date)
    daily.write.parquet(f"/output/{date}")
```

In [ ]:
# 练习 2 答案

answer_2 = """
场景 A: 不需要 Cache
  原因: 只有一个 Action (count)，Cache 没有复用机会
  额外: 500GB 数据无法放入内存，强行 Cache 会导致内存压力

场景 B: 应该使用 Cache!
  原因: df 被 3 个 Action 使用 (count, agg, groupBy+take)
  建议: df.cache() 在第一个 Action 前调用
  注意: 2GB 可以放入内存，Cache 效果好
  代码:
    df = spark.read.parquet('/data/users').cache()
    count = df.count()           # 触发缓存
    avg_age = df.agg(...)        # 从缓存读
    top_cities = df.groupBy(...) # 从缓存读
    df.unpersist()               # 用完释放

场景 C: 不应该对原始 df Cache
  原因: 100GB 无法放入内存
  更好的方案: 
    1. 使用分区裁剪: 按 date 分区存储，每次只读相关分区
    2. 重写为批量写出: df.write.partitionBy('date').parquet('/output')
    3. 如果必须循环，考虑 DPP 或 repartition
"""
print(answer_2)

### 练习 3：Join 策略选择

对于以下 Join 场景，选择最优策略并解释：

| 场景 | 左表 | 右表 | 过滤后 | 推荐策略 |
|------|------|------|--------|----------|
| A | 100GB | 5MB | 无过滤 | ? |
| B | 100GB | 2GB | 右表过滤后 8MB | ? |
| C | 500GB | 300GB | 无过滤 | ? |
| D | 100GB | 50GB | 左表倾斜 | ? |

In [ ]:
# 练习 3 答案

join_strategies = [
    {
        "场景": "A",
        "左表": "100GB",
        "右表": "5MB",
        "策略": "BroadcastHashJoin",
        "理由": "右表 5MB < 10MB 阈值，自动广播。大表无需 Shuffle",
        "代码": "join(broadcast(small_df), 'key')  # 或自动触发"
    },
    {
        "场景": "B",
        "左表": "100GB",
        "右表": "2GB→8MB",
        "策略": "AQE BroadcastHashJoin",
        "理由": "静态估计 2GB → SortMergeJoin。AQE 运行时发现过滤后 8MB，动态切换为 BroadcastHashJoin",
        "代码": "spark.conf.set('spark.sql.adaptive.enabled', 'true')  # 开启 AQE"
    },
    {
        "场景": "C",
        "左表": "500GB",
        "右表": "300GB",
        "策略": "SortMergeJoin + Bucketing",
        "理由": "两表都很大，无法广播。如果频繁 Join，建议 Bucketing 消除 Shuffle",
        "代码": "write.bucketBy(256, 'key').saveAsTable(...)  # 预分区"
    },
    {
        "场景": "D",
        "左表": "100GB (倾斜)",
        "右表": "50GB",
        "策略": "AQE Skew Join 或 手动 Salting",
        "理由": "数据倾斜导致长尾任务。AQE 可自动检测并处理，或手动实施 Salting",
        "代码": "spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')"
    },
]

print("=" * 60)
print("Join 策略选择分析")
print("=" * 60)
for s in join_strategies:
    print(f"\n场景 {s['场景']}: 左表({s['左表']}) JOIN 右表({s['右表']})")
    print(f"  推荐策略: {s['策略']}")
    print(f"  理由: {s['理由']}")
    print(f"  配置: {s['代码']}")

### 练习 4：数据倾斜处理

你有一个 Join 操作，发现某个 Task 处理了 95% 的数据，导致 Job 一直等待这个 Task。请描述完整的诊断和解决方案。

In [ ]:
# 练习 4 答案：完整 Salting 实现
import pandas as pd
import numpy as np

print("=" * 60)
print("数据倾斜诊断与 Salting 解决方案")
print("=" * 60)

print("""
诊断步骤:
  1. Spark UI → Stages → 查看 Task 耗时分布
  2. 找到耗时最长的 Task → 查看 Input Size
  3. 分析 Join Key 的数据分布:
""")

# 用 pandas 模拟诊断
np.random.seed(42)
SALT = 8  # SALT_FACTOR

# 生成倾斜的大表数据
n = 50_000
big_df = pd.DataFrame({
    'user_id': np.random.choice(
        [1] * 45000 + list(range(2, 101)) * (5000 // 99),  # user_id=1 占 90%
        n
    ),
    'amount': np.random.randint(1, 100, n)
})

small_df = pd.DataFrame({
    'user_id': range(1, 101),
    'name': [f'user_{i}' for i in range(1, 101)]
})

print("大表 Key 分布 (诊断倾斜):")
top_keys = big_df['user_id'].value_counts().head(5)
for uid, count in top_keys.items():
    pct = count / len(big_df) * 100
    bar = '█' * int(pct)
    print(f"  user_id={uid}: {bar} {count:,} ({pct:.1f}%)")

print()
print("解决方案 - Salting:")
print(f"  SALT_FACTOR = {SALT}")

# Salting 实现
# Step 1: 大表加 salt
big_df_salted = big_df.copy()
big_df_salted['salt'] = np.random.randint(0, SALT, len(big_df))
big_df_salted['salted_key'] = big_df_salted['user_id'].astype(str) + '_' + big_df_salted['salt'].astype(str)

# Step 2: 小表复制 SALT 份
small_df_exploded = pd.concat([
    small_df.assign(salt=i, salted_key=small_df['user_id'].astype(str) + '_' + str(i))
    for i in range(SALT)
])

# Step 3: Join
result = big_df_salted.merge(small_df_exploded[['salted_key', 'name']], on='salted_key')

# 验证 key 分布
print("\nSalting 后 Key 分布 (应该均匀):")
salted_dist = big_df_salted[big_df_salted['user_id'] == 1]['salt'].value_counts().sort_index()
for salt_val, count in salted_dist.items():
    bar = '█' * (count // 500)
    print(f"  user_1_salt_{salt_val}: {bar} {count:,}")

print(f"\n结果验证: {len(result):,} 行 (应等于大表行数 {len(big_df):,})")
print("热点 key 从单分区 45000 行 → 8个分区各约 5600 行")

### 练习 5：AQE 配置优化

你的 Spark Job 有以下问题：
1. Shuffle 后有 1000 个分区，但大部分很小（< 1MB）
2. 静态计划选择了 SortMergeJoin，但 Join 的右表实际很小
3. 某些分区特别大，导致长尾 Task

请写出解决以上三个问题的 AQE 配置。

In [ ]:
# 练习 5 答案

aqe_solution = '''
# 问题 1: 小分区太多 → 开启分区合并
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64mb")
# 效果: 将 1000 个小分区合并为 ~20 个 64MB 的分区

# 问题 2: 动态切换 Join 策略
spark.conf.set("spark.sql.adaptive.enabled", "true")
# AQE 会在 Shuffle 完成后，检测到右表实际只有 8MB
# 自动从 SortMergeJoin 切换为 BroadcastHashJoin
# 确保广播阈值设置合理:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50mb")

# 问题 3: 分区倾斜 → 开启 Skew Join 优化
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
# 分区大小超过中位数 5 倍才触发处理
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256mb")
# 分区大于 256MB 且满足 Factor 才判定倾斜

# 一次性配置（生产环境推荐）:
spark = SparkSession.builder \\
    .config("spark.sql.adaptive.enabled", "true") \\
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \\
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64mb") \\
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \\
    .config("spark.sql.autoBroadcastJoinThreshold", "50mb") \\
    .getOrCreate()
'''

print("AQE 配置解决方案:")
print(aqe_solution)

### 练习 6：综合架构设计

你需要设计一个每天处理 10TB 数据的 Spark 数仓，核心是将 1个 500GB 事实表与 5 个维度表（5MB~2GB）进行 Star Schema Join，然后按天聚合。请设计最优的执行策略。

In [ ]:
# 练习 6 答案

answer_6 = """
Star Schema 最优执行策略设计:

1. 存储层设计:
   事实表 (500GB): 按 date 分区存储 + 按常用 Join Key (product_id) Bucketing
     fact_sales.write
       .partitionBy("date")              # 分区裁剪: 按日期过滤
       .bucketBy(256, "product_id")      # 消除与大维度表的 Shuffle
       .saveAsTable("fact_sales")

2. 维度表处理策略:
   小维度表 (< 100MB): dim_date, dim_store → Broadcast Join
     自动广播或: join(broadcast(dim_date), "date_id")

   中维度表 (100MB~500MB): dim_product (如果常用 Join)
     → Bucketing: bucketBy(256, "product_id")
     → 与 fact 同 bucket 数 → 消除 Shuffle

   大维度表 (2GB): dim_user
     → SortMergeJoin (无法广播)
     → 开启 AQE: 自动处理倾斜、合并小分区

3. AQE 配置:
   spark.sql.adaptive.enabled = true
   spark.sql.adaptive.skewJoin.enabled = true    # 处理用户倾斜
   spark.sql.autoBroadcastJoinThreshold = 100mb  # 提高广播阈值

4. DPP 利用:
   WHERE date = '2024-01-01' → DPP 只扫描事实表对应日期分区
   10TB / 365天 ≈ 27GB 的日增量 → 每天只处理 27GB!

5. 执行计划:
   DPP 过滤事实表 (27GB)
   → Broadcast Join with dim_date (5MB)
   → Broadcast Join with dim_store (10MB)
   → Bucket Join with dim_product (无 Shuffle)
   → SortMerge Join with dim_user (+ AQE)
   → 聚合
   → 写出

6. 性能预期:
   vs 无优化 (纯 SortMergeJoin): ~8小时
   vs 上述优化: ~45分钟
   关键: DPP (减少 95% 扫描) + BroadcastJoin (消除 2 次 Shuffle) +
         Bucketing (消除 1 次 Shuffle) = 整体提速 10x+
"""
print(answer_6)